In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision

In [ ]:
# Binary segmentation: foreground digit pixels versus background.
# These are synthetic masks derived from MNIST intensities, not annotated masks.
class MNISTSegmentation(torch.utils.data.Dataset):
    def __init__(self, train):
        self.dataset = datasets.MNIST(root='./data_unet', train=train, download=True)
        self.resize = transforms.Resize((572, 572))
        self.normalize = transforms.Normalize((0.1307,), (0.3081,))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        image = transforms.ToTensor()(self.resize(image))
        # Threshold before normalization; crop to match the valid-convolution output.
        mask = (image[0] > 0.5).long()
        mask = transforms.functional.center_crop(mask, (388, 388))
        return self.normalize(image), mask

train_dataset = MNISTSegmentation(train=True)
test_dataset = MNISTSegmentation(train=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=(3,3))
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3,3))
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3,3))
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3,3))
        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3,3))
        self.conv6 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3))
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3))
        self.conv8 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3))
        self.conv9 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=(3,3))
        self.conv10 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3))
        self.conv_up1 = nn.ConvTranspose2d(in_channels=1024, out_channels=512, kernel_size=(2,2), stride=(2,2))
        self.conv11 = nn.Conv2d(in_channels=1024, out_channels=512, kernel_size=(3,3))
        self.conv12 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(3,3))
        self.conv_up2 = nn.ConvTranspose2d(in_channels=512, out_channels=256, kernel_size=(2,2), stride=(2,2))
        self.conv13 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=(3,3))
        self.conv14 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(3,3))
        self.conv_up3 = nn.ConvTranspose2d(in_channels=256, out_channels=128, kernel_size=(2,2), stride=(2,2))
        self.conv15 = nn.Conv2d(in_channels=256, out_channels=128, kernel_size=(3,3))
        self.conv16 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3,3))
        self.conv_up4 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(2,2), stride=(2,2))
        self.conv17 = nn.Conv2d(in_channels=128, out_channels=64, kernel_size=(3,3))
        self.conv18 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3,3))
        self.conv19 = nn.Conv2d(in_channels=64, out_channels=2, kernel_size=(1,1))
        self.max_pool = nn.MaxPool2d(kernel_size=(2,2))

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x_copy1 = x
        x = self.max_pool(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x_copy2 = x
        x = self.max_pool(x)
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x_copy3 = x
        x = self.max_pool(x)
        x = F.relu(self.conv7(x))
        x = F.relu(self.conv8(x))
        x_copy4 = x
        x = self.max_pool(x)
        x = F.relu(self.conv9(x))
        x = F.relu(self.conv10(x))
        x = self.conv_up1(x)
        x_crop4 = torchvision.transforms.CenterCrop(56)(x_copy4)
        x = torch.cat((x_crop4, x), 1)
        x = F.relu(self.conv11(x))
        x = F.relu(self.conv12(x))
        x = self.conv_up2(x)
        x_crop3 = torchvision.transforms.CenterCrop(104)(x_copy3)
        x = torch.cat((x_crop3, x), 1)
        x = F.relu(self.conv13(x))
        x = F.relu(self.conv14(x))
        x = self.conv_up3(x)
        x_crop2 = torchvision.transforms.CenterCrop(200)(x_copy2)
        x = torch.cat((x_crop2, x), 1)
        x = F.relu(self.conv15(x))
        x = F.relu(self.conv16(x))
        x = self.conv_up4(x)
        x_crop1 = torchvision.transforms.CenterCrop(392)(x_copy1)
        x = torch.cat((x_crop1, x), 1)
        x = F.relu(self.conv17(x))
        x = F.relu(self.conv18(x))
        x = self.conv19(x)
        return x

model = UNet()
device ='cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
# Defining Loss Function (Since it is a multi-class classification, I chose CrossEntropyLoss)
loss_fn = nn.CrossEntropyLoss()
# Defining Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-7, weight_decay=1e-4)

In [ ]:
# Metrics are calculated over pixels, not image labels.
def run_epoch(loader, training):
    model.train(training)
    loss_total = 0.0
    pixel_total = 0
    correct_total = 0
    intersection_total = 0
    union_total = 0
    foreground_pred_total = 0
    foreground_true_total = 0

    with torch.set_grad_enabled(training):
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            if logits.shape[0] != masks.shape[0] or logits.shape[2:] != masks.shape[1:]:
                raise ValueError(f"Output/mask shape mismatch: {logits.shape}, {masks.shape}")
            loss = loss_fn(logits, masks)
            if training:
                loss.backward()
                optimizer.step()

            predictions = logits.detach().argmax(dim=1)
            pixels = masks.numel()
            loss_total += loss.item() * pixels
            pixel_total += pixels
            correct_total += (predictions == masks).sum().item()
            pred_fg = predictions == 1
            true_fg = masks == 1
            intersection_total += (pred_fg & true_fg).sum().item()
            union_total += (pred_fg | true_fg).sum().item()
            foreground_pred_total += pred_fg.sum().item()
            foreground_true_total += true_fg.sum().item()

    foreground_total = foreground_pred_total + foreground_true_total
    return {
        'loss': loss_total / pixel_total,
        'accuracy': 100.0 * correct_total / pixel_total,
        'iou': intersection_total / union_total if union_total else 1.0,
        'dice': 2.0 * intersection_total / foreground_total if foreground_total else 1.0,
    }

# Foreground IoU/Dice also track segmentation quality when background dominates.
epochs = 100
history = []
for epoch in range(epochs):
    train_metrics = run_epoch(train_loader, training=True)
    test_metrics = run_epoch(test_loader, training=False)
    history.append({'epoch': epoch + 1, 'train': train_metrics, 'test': test_metrics})
    print(
        f"Epoch: {epoch + 1} | "
        f"Train loss: {train_metrics['loss']:.4f} | Test loss: {test_metrics['loss']:.4f} | "
        f"Train pixel acc: {train_metrics['accuracy']:.2f}% | "
        f"Test pixel acc: {test_metrics['accuracy']:.2f}% | "
        f"Test foreground IoU: {test_metrics['iou']:.4f} | "
        f"Test foreground Dice: {test_metrics['dice']:.4f}"
    )


In [ ]:
torch.save(model.state_dict(), 'unet.pth')